# LeWM Training
Run cells top to bottom. Checkpoints save to Google Drive after each epoch.

In [15]:
!pip uninstall -y mamba-ssm causal-conv1d
!pip install causal-conv1d --no-build-isolation
!pip install mamba-ssm --no-build-isolation

  Using cached causal_conv1d-1.6.2.post1.tar.gz (29 kB)
  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.2.post1-cp312-cp312-linux_x86_64.whl size=171314099 sha256=240479acc522c8fa6d91fe99a308e9a7d7c2c63174ea41bb57a40e2e942a75f8
  Stored in directory: /root/.cache/pip/wheels/ea/f0/26/5d87ae05a302e6dc8016c50cd8c7ee779585f593b9580e7cf8
Successfully built causal-conv1d
  Using cached mamba_ssm-2.3.2.post1.tar.gz (216 kB)
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.4/358.4 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.7/767.7 kB 62.8 MB/s eta 0:0

In [1]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, os, sys, re, threading, shutil, glob, time
subprocess.run(['pip', 'install', '-q', 'stable-worldmodel[train]'], check=True)
print('Done')

Done


In [2]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CKPT = '/content/drive/MyDrive/lewm_checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_CKPT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints will be saved to: /content/drive/MyDrive/lewm_checkpoints


In [3]:
# ── Clone repo + patch config ─────────────────────────────────────────────────
if not os.path.exists('/content/le-wm'):
    subprocess.run(['git', 'clone', 'https://github.com/UnnatPar/lewm-jamba', '/content/le-wm'], check=True)

config_path = '/content/le-wm/config/train/data/pusht.yaml'
with open(config_path) as f:
    cfg = f.read()
cfg = re.sub(r'pusht_expert_train(?:\.lance)?(?:\.h5)?', 'pusht_expert_train.lance', cfg)
with open(config_path, 'w') as f:
    f.write(cfg)
print('Repo ready')

Repo ready


In [8]:
# ── Download dataset (~12 GB, ~10 min) ───────────────────────────────────────
h5_path  = '/content/stable-wm/datasets/pusht_expert_train.h5'
zst_path = h5_path + '.zst'
os.makedirs('/content/stable-wm/datasets', exist_ok=True)

if not os.path.exists(h5_path):
    dl = subprocess.Popen(
        ['wget', '--progress=dot:giga', '-O', zst_path,
         'https://huggingface.co/datasets/quentinll/lewm-pusht/resolve/main/pusht_expert_train.h5.zst'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in iter(dl.stdout.readline, b''):
        sys.stdout.write(line.decode(errors='replace')); sys.stdout.flush()
    dl.wait()
    if dl.returncode != 0:
        raise SystemExit('wget failed')

    subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'], check=True)
    print('Decompressing (~5 min)...')
    subprocess.run(['zstd', '-d', zst_path, '-o', h5_path, '--rm'], check=True)

size_gb = os.path.getsize(h5_path) / 1e9
print(f'H5 ready: {size_gb:.1f} GB')
if size_gb < 30:
    raise SystemExit(f'H5 too small — download incomplete')

--2026-07-13 20:53:33--  https://huggingface.co/datasets/quentinll/lewm-pusht/resolve/main/pusht_expert_train.h5.zst
Resolving huggingface.co (huggingface.co)... 13.35.202.97, 13.35.202.121, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.97|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/69c59e11826fe701d9e91858/a8dcf14ef922f03e388acda1ac87a643d0684d17ef0374f6f08bc963f3c7e58f?Expires=1783979613&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY5YzU5ZTExODI2ZmU3MDFkOWU5MTg1OC9hOGRjZjE0ZWY5MjJmMDNlMzg4YWNkYTFhYzg3YTY0M2QwNjg0ZDE3ZWYwMzc0ZjZmMDhiYzk2M2YzYzdlNThmKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4Mzk3OTYxM319fV19&Signature=MEYCIQCrWvGSa-Mix9yvGRSLCieQPnEKBFw1MiVtlnY%7EVpg-ZAIhALllnSoDXsUIqnKgW6PiMJ-szz%7EvGAgJUYKK0xlOkxuG&Key-Pair-Id=K1LYXO563TGWFU&X-Xet-Cas-Uid=public&response-content-disposition=in

In [9]:
# ── Convert H5 → Lance (~20 min, one-time) ────────────────────────────────────
lance_path = '/content/stable-wm/datasets/pusht_expert_train.lance'

lance_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(lance_path) for f in files
) if os.path.exists(lance_path) else 0

if lance_size < 1_000_000_000:
    if os.path.exists(lance_path):
        shutil.rmtree(lance_path)

    subprocess.run(['pip', 'install', '-q', 'hdf5plugin'], check=True)
    import hdf5plugin, h5py
    from tqdm.notebook import tqdm
    from stable_worldmodel.data.format import get_format

    with h5py.File(h5_path, 'r') as f:
        ep_offsets = f['ep_offset'][:]
        ep_lens    = f['ep_len'][:]
        n_eps      = len(ep_lens)
    print(f'{n_eps} episodes')

    def episodes():
        with h5py.File(h5_path, 'r') as hf:
            for i in tqdm(range(n_eps), desc='converting'):
                o, l = int(ep_offsets[i]), int(ep_lens[i])
                yield {k: list(hf[k][o:o+l]) for k in ['pixels','action','proprio','state']}

    with get_format('lance').open_writer(lance_path, mode='overwrite') as w:
        w.write_episodes(episodes())
    print('Lance ready')
else:
    print(f'Lance already complete ({lance_size/1e9:.1f} GB)')

18685 episodes


converting:   0%|          | 0/18685 [00:00<?, ?it/s]

Lance ready


In [9]:
# ── Restore checkpoint from Drive (skip if starting fresh) ────────────────────
import shutil, os, glob

# pick the latest epoch saved to Drive
drive_ckpts = sorted(glob.glob('/content/drive/MyDrive/lewm_checkpoints/weights_epoch_*.ckpt'))
if drive_ckpts:
    latest = drive_ckpts[-1]
    dst = '/content/stable-wm/checkpoints/lewm_pusht/lewm_weights.ckpt'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(latest, dst)
    print(f'Restored: {os.path.basename(latest)} → {dst}  ({os.path.getsize(dst)/1e6:.0f} MB)')
else:
    print('No Drive checkpoint found — will train from scratch')

No Drive checkpoint found — will train from scratch


In [7]:
import subprocess, os, sys, re, threading, shutil, glob, time

# ── Train 10 epochs (checkpoints auto-save to Drive after each epoch) ─────────
CKPT_OUT = '/content/stable-wm/checkpoints'
os.makedirs(CKPT_OUT, exist_ok=True)
_stop = threading.Event()
_seen = set()

def watcher():
    while not _stop.is_set():
        for path in glob.glob('/root/.cache/stable-pretraining/runs/*/*/*/checkpoints/epoch=*.ckpt'):
            if path in _seen:
                continue
            m = re.search(r'epoch=(\d+)', path)
            if not m:
                continue
            try:
                s1 = os.path.getsize(path); time.sleep(5); s2 = os.path.getsize(path)
                if s1 != s2 or s1 == 0:
                    continue
            except OSError:
                continue
            epoch = m.group(1)
            fname = f'weights_epoch_{epoch}.ckpt'
            # copy to local predictable path
            local = os.path.join(CKPT_OUT, fname)
            shutil.copy2(path, local)
            # copy to Drive
            drive_dst = os.path.join(DRIVE_CKPT, fname)
            shutil.copy2(path, drive_dst)
            _seen.add(path)
            size_mb = os.path.getsize(local) / 1e6
            print(f'=== CHECKPOINT SAVED: {fname} ({size_mb:.0f} MB) → Drive + {CKPT_OUT} ===', flush=True)

        _stop.wait(30)

threading.Thread(target=watcher, daemon=True).start()

env = os.environ.copy()
env['LOCAL_DATASET_DIR'] = '/content/stable-wm'
env['STABLEWM_HOME']     = '/content/stable-wm'
env['PYTHONUNBUFFERED']  = '1'

proc = subprocess.Popen(
    ['python', '-u', 'train.py',
     'subdir=lewm_pusht',
     'loader.batch_size=128',
     'num_workers=4',
     '+loader.multiprocessing_context=spawn',
     'trainer.max_epochs=11'],
    cwd='/content/le-wm', env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
for line in iter(proc.stdout.readline, b''):
    sys.stdout.write(line.decode(errors='replace')); sys.stdout.flush()
proc.wait()
_stop.set()

if proc.returncode != 0:
    raise SystemExit(f'train.py failed with code {proc.returncode}')
print('=== TRAINING COMPLETE ===')


22:49:55 | INFO  | __init__.py | TensorFlow version 2.20.0 available.
22:49:55 | INFO  | __init__.py | JAX version 0.7.2 available.
22:49:59 | INFO  | atomic_chec~| [atomic_save] installed crash-safe checkpoint plugin (write to sibling .tmp + fsync + atomic rename)
[2026-07-13 22:49:59,329][root][WARNING] - LanceDataset: keys_to_cache=['action', 'proprio', 'state'] is not required — Lance has efficient random access via batched __getitems__.
22:50:33 | INFO  | module.py   | Setting up DataModule
22:50:33 | INFO  | utils.py    | ── Module ────────────────────────────────────────
22:50:33 | WARN  | module.py   | ! No hyperparameters provided - hyperparameter logging is disabled.
22:50:33 | INFO  | module.py   |   Setting custom forward method.
22:50:33 | INFO  | module.py   |   Setting attribute: self.model = <class 'jepa.JEPA'>
22:50:33 | INFO  | module.py   |   Setting attribute: self.sigreg = <class 'module.SIGReg'>
22:50:33 | INFO  | module.py   |   Setting attribute: self.optim = <c

KeyboardInterrupt: 

In [6]:
import subprocess, os, sys, re, shutil, glob, json
subprocess.run(['pip', 'install', '-q', 'hdf5plugin'], check=True)
subprocess.run(['pip', 'install', '-q', 'pymunk'], check=True)
import torch
import hdf5plugin, h5py
import stable_worldmodel  # registers custom OmegaConf resolvers used by config/train/lewm.yaml
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf, open_dict

# ── Evaluate a trained checkpoint from Drive, save results to Drive ──────────
DRIVE_RESULTS = '/content/drive/MyDrive/lewm_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# pick the checkpoint with the highest epoch number already sitting in Drive
drive_ckpts = sorted(
    glob.glob(f'{DRIVE_CKPT}/weights_epoch_*.ckpt'),
    key=lambda p: int(re.search(r'epoch_(\d+)', p).group(1)),
)
if not drive_ckpts:
    raise SystemExit(f'No checkpoint found in {DRIVE_CKPT}')
latest_ckpt = drive_ckpts[-1]
print(f'Evaluating checkpoint: {os.path.basename(latest_ckpt)}')

# ── rebuild the model config exactly as train.py does (same defaults, same
#    dynamically-computed action_encoder.input_dim), then lift the model's
#    weights out of the Lightning checkpoint. This gives eval.py's
#    load_pretrained the (weights.pt + config.json) pair it expects, without
#    needing anything extra saved during training. ──────────────────────────
with initialize_config_dir(config_dir='/content/le-wm/config/train', version_base=None):
    train_cfg = compose(config_name='lewm')

with h5py.File('/content/stable-wm/datasets/pusht_expert_train.h5', 'r') as f:
    action_dim = f['action'].shape[-1]
with open_dict(train_cfg):
    train_cfg.model.action_encoder.input_dim = train_cfg.data.dataset.frameskip * action_dim
model_cfg = OmegaConf.to_container(train_cfg.model, resolve=True)

ckpt = torch.load(latest_ckpt, map_location='cpu', weights_only=False)
raw_sd = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
state_dict = {k[len('model.'):]: v for k, v in raw_sd.items() if k.startswith('model.')}
if not state_dict:
    state_dict = raw_sd  # already a bare model state_dict

EVAL_RUN_NAME = 'eval_' + os.path.splitext(os.path.basename(latest_ckpt))[0]  # e.g. eval_weights_epoch_5
eval_ckpt_dir = f'/content/stable-wm/checkpoints/{EVAL_RUN_NAME}'
os.makedirs(eval_ckpt_dir, exist_ok=True)
torch.save(state_dict, os.path.join(eval_ckpt_dir, 'weights.pt'))
with open(os.path.join(eval_ckpt_dir, 'config.json'), 'w') as f:
    json.dump(model_cfg, f, indent=2)

env = os.environ.copy()
env['STABLEWM_HOME']    = '/content/stable-wm'
env['PYTHONUNBUFFERED'] = '1'

proc = subprocess.Popen(
    ['python', '-u', 'eval.py', f'policy={EVAL_RUN_NAME}'],
    cwd='/content/le-wm', env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
for line in iter(proc.stdout.readline, b''):
    sys.stdout.write(line.decode(errors='replace')); sys.stdout.flush()
proc.wait()

if proc.returncode != 0:
    raise SystemExit(f'eval.py failed with code {proc.returncode}')

# ── copy results (metrics txt + rollout videos) to Drive ─────────────────────
results_dir = os.path.join(DRIVE_RESULTS, EVAL_RUN_NAME)
os.makedirs(results_dir, exist_ok=True)

local_results_txt = '/content/stable-wm/pusht_results.txt'
if os.path.exists(local_results_txt):
    shutil.copy2(local_results_txt, results_dir)

for video in glob.glob('/content/stable-wm/*.mp4'):
    shutil.copy2(video, results_dir)

print(f'=== EVAL COMPLETE — results')

SystemExit: No checkpoint found in /content/drive/MyDrive/lewm_checkpoints

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
